# 🚀 Evaluation Metrics for LLM & RAG Systems: Theory and Practice

## From Traditional Metrics to Modern Evaluation Frameworks! 📊

This notebook provides a **comprehensive guide** to evaluation metrics — starting from classical Information Retrieval (IR) metrics and building up to modern LLM/RAG evaluation frameworks.

### 🎯 What You'll Learn

By the end of this lab, you'll master:

1. **📚 Traditional Metrics** - Precision, Recall, F1-Score and why they're not enough for LLMs
2. **🔍 RAG-Specific Metrics** - Contextual Precision/Recall, Answer Relevancy, Faithfulness
3. **🧪 Hands-On Implementation** - Code Precision@k, Recall@k, AP@k, and nDCG from scratch
4. **📊 Evaluation Frameworks** - RAGAS, G-Eval, DAG evaluation, and Agentic metrics
5. **🧠 When to Use What** - Choosing the right metric for your use case

### 🌟 Why This Matters

Evaluation is the **most critical** yet often overlooked part of building AI systems:
- **Without evaluation**, you're flying blind — you don't know if your system is actually working
- **Wrong metrics** can give you false confidence — a system can score high on Precision but still hallucinate
- **Modern frameworks** like RAGAS and G-Eval go beyond simple retrieval to evaluate the full pipeline

### 📋 Notebook Structure:
- **Part 1**: 📚 Theory — comprehensive coverage of all evaluation metric categories
- **Part 2**: 💻 Practice — implement and run classical IR metrics on example data

---

*💡 **Pro Tip**: Understanding classical metrics deeply makes it much easier to appreciate why modern frameworks like RAGAS were created — they address specific gaps that classical metrics can't fill!*

# Part 1: 📚 Evaluation Metrics in LLM and RAG Systems — Theory & Concepts

---

## 1. Traditional Metrics: Precision, Recall, F1-Score

In classical Machine Learning (e.g., classification tasks), we often rely on **Precision, Recall, and F1-score** as the core evaluation metrics.

- **Precision**: Out of the predicted positives, how many are correct?  
  *Example*: A RAG system retrieved 10 passages, 6 are relevant.  
  Precision = 6/10 = **0.6**

- **Recall**: Out of all relevant documents, how many did we retrieve?  
  *Example*: There were 8 relevant passages in the corpus, we retrieved 6.  
  Recall = 6/8 = **0.75**

- **F1-Score**: The harmonic mean of precision and recall, balancing the two.  
  *Example*: Using the above numbers,  
  F1 = 2 × (0.6 × 0.75) / (0.6 + 0.75) ≈ **0.67**

---

### ❌ Why Precision/Recall/F1 Are Not Enough?

Precision, Recall, and F1-score are excellent for document retrieval tasks but **fall short for LLM and RAG-based systems**, where both retrieval quality and answer generation quality matter.  
They fail to capture dimensions that go beyond simple document relevance:

#### 🔹 Faithfulness (Groundedness to Retrieved Context)
- **What it means**: Does the generated answer stay true to the retrieved context, or does it hallucinate information?  
- **Why they fail**: They only check retrieval, not whether the answer used them correctly.  
- *Example*:  
  - Retrieved doc: *“The capital of France is Paris.”*  
  - Generated answer: *“The capital of France is Lyon.”*  
  - Retrieval scores → High  
  - Actual answer → **Unfaithful / hallucinated**

#### 🔹 Relevance of the Final Response
- **What it means**: Does the final generated response actually answer the user’s query directly and completely?  
- **Why they fail**: They stop at retrieval and don’t measure final answer quality.  
- *Example*:  
  - Query: *“What year was the iPhone first released?”*  
  - Retrieved doc: *“Apple released the first iPhone in 2007.”*  
  - Answer A: *“The iPhone was first released in 2007.”* → ✅ Relevant  
  - Answer B: *“Apple is a famous tech company.”* → ❌ Not relevant  
  - Precision/Recall → Same for both cases, but only Answer A is useful.

#### 🔹 Conversational Dynamics (Multi-turn Interaction)
- **What it means**: In multi-turn conversations, the model must track context, follow its role, and stay coherent.  
- **Why they fail**: They are single-turn metrics and cannot evaluate flow.  
- *Example*:  
  - User (turn 1): *“Who is the Prime Minister of India?”*  
    Model: *“Narendra Modi is the Prime Minister of India.”*  
  - User (turn 2): *“What was his role before becoming PM?”*  
    - Correct answer: *“He served as the Chief Minister of Gujarat from 2001 to 2014.”*  
    - Incorrect: *“I don’t know who you mean by ‘he’.”*  
  - Precision/Recall/F1 → Cannot score this.  

➡️ This is where **advanced evaluation frameworks like RAGAS and G-Eval** step in.

---

## 2. RAG-Specific Metrics

Retrieval-Augmented Generation (RAG) involves two steps:
1. **Retrieving documents (Information Retrieval)**
2. **Generating an answer using those documents (Response Generation)**

So, evaluation must cover both retrieval quality and answer quality.

---

### 🔹 Contextual Precision
**Definition**: Measures how many of the retrieved contexts are actually relevant to the query.

\[
\text{Contextual Precision} = \frac{\text{Relevant Retrieved Contexts}}{\text{Total Retrieved Contexts}}
\]

- **Factors affecting score**:
  - Size of retrieval set (large set lowers precision).
  - Noise in pipeline (irrelevant passages).
  - Embedding/keyword quality.

- **Example**:  
  Query: *“Who discovered penicillin?”*  
  Retrieved 5 passages → 4 relevant, 1 irrelevant.  
  Precision = 4/5 = **0.8**

---

### 🔹 Contextual Recall
**Definition**: Measures how many of the total relevant contexts were retrieved.

\[
\text{Contextual Recall} = \frac{\text{Relevant Retrieved Contexts}}{\text{Total Relevant Contexts in Corpus}}
\]

- **Factors affecting score**:
  - Small top-k lowers recall.
  - Missing semantically close docs.
  - Poor query expansion.

- **Example**:  
  10 relevant passages exist, system retrieved 4.  
  Recall = 4/10 = **0.4**

---

### 🔹 Contextual Relevancy
**Definition**: Evaluates whether retrieved docs are semantically useful, not just keyword overlap.

- **Factors**:
  - Semantic alignment.
  - Actual support for answering.

- **Example**:  
  Query: *“Who discovered penicillin?”*  
  - Doc A: *“Alexander Fleming discovered penicillin in 1928.”* → ✅ Relevant  
  - Doc B: *“Penicillin is an antibiotic.”* → ❌ Low relevance  
  Relevancy = 1/2 = **0.5**

---

### 🔹 Answer Relevancy
**Definition**: Checks whether the generated answer addresses the user query.

\[
\text{Answer Relevancy} = \frac{\text{Relevant Answer Segments}}{\text{Total Answer Segments}}
\]

- **Factors**:
  - Completeness.  
  - Staying on-topic.  
  - Conciseness.  

- **Example**:  
  - Query: *“Capital of France?”*  
  - Answer: *“Paris is the capital of France.”* → ✅ Relevant (1)  
  - Answer: *“France is in Europe.”* → ❌ Not relevant (0)  
  Avg. relevancy = **0.5**

---

### 🔹 Faithfulness
**Definition**: Whether the generated answer is factually consistent with retrieved context.

- **Factors**:
  - Hallucination.  
  - Misinterpretation.  
  - Over-generalization.  

- **Example**:  
  - Context: *“The capital of France is Paris.”*  
  - Answer A: *“The capital of France is Lyon.”* → ❌ Unfaithful (0)  
  - Answer B: *“Paris is the capital of France.”* → ✅ Faithful (1)

---

## 3. RAGAS (Retrieval-Augmented Generation Assessment)

RAGAS provides a **unified framework** by combining retrieval + generation metrics.  
It focuses on **semantic evaluation**, not just exact matches.  
It uses **LLM-based judges** to score:
- Contextual Precision  
- Contextual Recall  
- Answer Relevance  
- Faithfulness  

**Example**:  
- Query: *“When was the iPhone first released?”*  
- Context: *“Apple launched the first iPhone in 2007.”*  
- Answer: *“The iPhone was first released in 2007 by Apple.”*  

✅ High contextual precision  
✅ High faithfulness  
✅ High answer relevancy  

---

## 4. G-Eval: LLM-as-a-Judge

- Uses **LLM as a judge** with structured prompts.  
- Great for open-ended answers where BLEU/ROUGE fail.  
- Evaluates: **fluency, coherence, factuality, style**.  

*Example*: Summarization task – instead of exact match, G-Eval checks if **all key points are covered**.

---

## 5. DAG (Directed Acyclic Graph) for Multi-Metric Evaluation

- **Nodes** = metrics (precision, recall, faithfulness, etc.)  
- **Edges** = dependencies (e.g., faithfulness depends on retrieval quality).  
- Ensures logical flow from retrieval → reasoning → generation.

---

## 6. Agentic Metrics (for Agents and Tool Use)

When LLMs act as **agents** (tools, APIs, multi-step reasoning):

- **Task Completion**: Did the agent achieve the end goal?  
  *Example*: Successfully booked a flight.  
- **Tool Correctness**: Was the right tool/API used correctly?  
  *Example*: Used weather API vs mistakenly calling calendar API.

---

## 7. Other Critical Metrics

- **Hallucination**: Checks fabricated info.  
- **Summarization Quality**: Are key points captured?  
- **Bias**: Detects unintended bias (gender, cultural, political).  
- **Toxicity**: Detects unsafe or offensive output.

---

## 8. Conversational Metrics (for Multi-turn Chat)

- **Knowledge Retention**: Remembers past turns.  
- **Conversation Completeness**: Covers all aspects of query.  
- **Conversation Relevancy**: Stays on-topic.  
- **Role Adherence**: Follows its assigned role.  

*Example*:  
- User: *“Explain Newton’s first law.”*  
  - Assistant: *“Newton’s first law states that…”* → ✅ Role adhered  
  - Assistant: *“I don’t want to talk about physics.”* → ❌ Role violation  

---

## ✅ Final Takeaway

- Precision, Recall, F1 are **necessary but insufficient** for LLM/RAG evaluation.  
- **RAGAS** → semantic evaluation (retrieval + generation).  
- **G-Eval** → LLM-as-judge for nuanced outputs.  
- **Agentic & Conversational metrics** → real-world scenarios.  

A **holistic evaluation** must combine:
- Traditional IR metrics (Precision, Recall, F1)  
- RAG metrics (Faithfulness, Contextual Precision/Recall, Relevance)  
- Agentic & Conversational metrics (Task Completion, Role Adherence, Hallucination, Bias, etc.)  

👉 Only then can we **trust LLMs and RAG systems in production**.


---

# Part 2: 💻 Hands-On Implementation — Classical IR Metrics

## Computing Precision@k, Recall@k, AP@k, and nDCG From Scratch 🧮

Now let's move from theory to practice! We'll implement the core Information Retrieval metrics manually to understand exactly how they work.

### 🔑 Metrics We'll Implement:

| Metric | What It Measures | Formula Intuition |
|---|---|---|
| **Precision@k** | Of the top-k retrieved items, how many are relevant? | relevant_in_top_k / k |
| **Recall@k** | Of all relevant items, how many appear in the top-k? | relevant_in_top_k / total_relevant |
| **AP@k (Average Precision)** | Precision averaged at each relevant position — rewards ranking relevant docs higher | Mean of precision values at relevant positions |
| **nDCG@k** | Normalized Discounted Cumulative Gain — handles graded relevance (not just binary) | Accounts for relevance scores AND position |
| **MAP@k** | Mean Average Precision — AP@k averaged across all queries | Mean of AP@k values |

### 📋 Our Test Data:
We'll use a **bakery-themed example** with three queries:
- **"Sweet pastry"** — looking for sweet items (Donut, Muffin, Scone are relevant)
- **"Suitable for lunch"** — looking for lunch items (Sandwich, Bagel, Roll, Pretzel are relevant)
- **"Goes well with jam"** — graded relevance (Bagel/Croissant/Roll are highly relevant, Scone/Muffin/Donut are somewhat relevant)

### 💡 Why Implement From Scratch?
Understanding the math behind these metrics helps you:
- **Debug** evaluation pipelines when scores seem wrong
- **Choose** the right metric for your use case (binary vs. graded relevance)
- **Appreciate** why nDCG is preferred over Precision@k for ranked retrieval

In [ ]:
import math
import statistics

# ============================================================
# 📋 STEP 1: Define Ground Truth (qrels) and System Results (runs)
# ============================================================
# "qrels" = query relevance judgments — the ground truth for each query
# Each query maps to a dict of {document: relevance_score}
# Binary relevance: 1 = relevant, 0 = not relevant
# Graded relevance: 2 = highly relevant, 1 = somewhat relevant, 0 = not relevant

qrels = {
    # 🍩 Binary relevance: Donut, Muffin, and Scone are sweet pastries
    'Sweet pastry': {
        'Donut': 1, 'Muffin': 1, 'Scone': 1
    },
    # 🥪 Binary relevance: Sandwich, Bagel, Roll, and Pretzel are lunch items
    'Suitable for lunch': {
        'Sandwich': 1, 'Bagel': 1, 'Roll': 1, 'Pretzel': 1
    },
    # 🍯 Graded relevance: Some items go BETTER with jam than others
    # 2 = highly relevant (Bagel, Croissant, Roll)
    # 1 = somewhat relevant (Scone, Muffin, Donut)
    'Goes well with jam': {
        'Bagel': 2, 'Croissant': 2, 'Roll': 2,
        'Scone': 1, 'Muffin': 1, 'Donut': 1
    },
}

# "runs" = the system's ranked retrieval results
# Each list is ordered by rank: index 0 = rank 1 (best), index 1 = rank 2, etc.
# This simulates what a retrieval system would return for each query

runs = {
    # 🍩 For "Sweet pastry": system returned Croissant first (wrong!), then Donut, Scone, Sandwich, Muffin
    'Sweet pastry': ['Croissant', 'Donut', 'Scone', 'Sandwich', 'Muffin'],
    # 🥪 For "Suitable for lunch": system returned Muffin first (wrong!), then Donut (wrong!), then 3 correct
    'Suitable for lunch': ['Muffin', 'Donut', 'Sandwich', 'Bagel', 'Roll'],
    # 🍯 For "Goes well with jam": system returned Pretzel first (not relevant!), then some relevant items
    'Goes well with jam': ['Pretzel', 'Bagel', 'Muffin', 'Donut', 'Croissant'],
}


# ============================================================
# 🧮 STEP 2: Implement Metric Functions
# ============================================================

def precision_at_k(retrieved, qrel, k):
    """
    Precision@k: What fraction of the top-k retrieved items are relevant?
    
    Formula: P@k = (number of relevant items in top-k) / k
    
    Example: If k=4 and 2 of the top-4 are relevant → P@4 = 2/4 = 0.5
    """
    # Create a binary relevance list: 1 if document is relevant, 0 if not
    rels = [1 if qrel.get(doc, 0) > 0 else 0 for doc in retrieved[:k]]
    return sum(rels) / k  # Count of relevant items divided by k


def recall_at_k(retrieved, qrel, k):
    """
    Recall@k: What fraction of ALL relevant items appear in the top-k?
    
    Formula: R@k = (number of relevant items in top-k) / (total relevant items in corpus)
    
    Example: If there are 5 relevant docs total and 3 appear in top-k → R@k = 3/5 = 0.6
    """
    # Create a binary relevance list for the top-k retrieved items
    rels = [1 if qrel.get(doc, 0) > 0 else 0 for doc in retrieved[:k]]
    # Count total number of relevant documents in the ground truth
    total_relevant = sum(1 for v in qrel.values() if v > 0)
    return sum(rels) / total_relevant if total_relevant > 0 else 0.0


def average_precision_at_k(retrieved, qrel, k):
    """
    Average Precision@k (AP@k): Rewards systems that rank relevant documents HIGHER.
    
    Formula: AP@k = (1/R) * sum_{i=1}^{k} P(i) * rel(i)
    where R = total relevant docs, P(i) = precision at position i, rel(i) = 1 if position i is relevant
    
    Key insight: AP@k is HIGHER when relevant documents appear at the TOP of the ranked list.
    Two systems with the same P@k can have very different AP@k values!
    """
    # Create a binary relevance list for the top-k retrieved items
    rels = [1 if qrel.get(doc, 0) > 0 else 0 for doc in retrieved[:k]]
    precisions_at_relevant = []  # Collect precision values at each relevant position
    num_relevant_found = 0

    for i, r in enumerate(rels, start=1):  # i starts at 1 (rank position)
        if r == 1:  # If the document at this position is relevant
            num_relevant_found += 1
            # Record the precision at this position (relevant found so far / position)
            precisions_at_relevant.append(num_relevant_found / i)

    # Normalize by total number of relevant documents in the corpus
    total_relevant = sum(1 for v in qrel.values() if v > 0)
    return sum(precisions_at_relevant) / total_relevant if total_relevant > 0 else 0.0


# ============================================================
# 📊 STEP 3: Implement nDCG (Normalized Discounted Cumulative Gain)
# ============================================================
# nDCG handles GRADED relevance (not just binary 0/1).
# It gives more credit to highly relevant documents and penalizes
# relevant documents that appear lower in the ranking.

def dcg_at_k(retrieved, qrel, k):
    """
    Discounted Cumulative Gain@k: Sum of relevance scores, discounted by log(position).
    
    Formula: DCG@k = sum_{i=1}^{k} (2^rel(i) - 1) / log2(i + 1)
    
    The log2 denominator means items at higher positions contribute less ("discount").
    The 2^rel numerator means higher relevance scores contribute exponentially more.
    """
    # Get the graded relevance score for each retrieved document (0 if not in qrel)
    rels = [qrel.get(doc, 0) for doc in retrieved[:k]]
    dcg = 0.0
    for i, rel in enumerate(rels, start=1):  # i starts at 1 (rank position)
        # Gain = (2^relevance - 1), Discount = log2(position + 1)
        dcg += (2**rel - 1) / math.log2(i + 1)
    return dcg


def idcg_at_k(qrel, k):
    """
    Ideal DCG@k: The BEST possible DCG if we ranked all relevant docs perfectly.
    
    This is computed by sorting all relevant documents by relevance (highest first)
    and computing DCG on this ideal ranking. Used as the denominator in nDCG.
    """
    # Sort all relevance scores in descending order and take top-k
    ideal_rels = sorted([v for v in qrel.values() if v > 0], reverse=True)[:k]
    idcg = 0.0
    for i, rel in enumerate(ideal_rels, start=1):
        idcg += (2**rel - 1) / math.log2(i + 1)
    return idcg


def ndcg_at_k(retrieved, qrel, k):
    """
    Normalized DCG@k: DCG normalized by the ideal DCG, giving a 0-1 score.
    
    Formula: nDCG@k = DCG@k / IDCG@k
    
    - nDCG = 1.0 means PERFECT ranking (all relevant docs at the top, ordered by relevance)
    - nDCG = 0.0 means no relevant docs were retrieved
    - nDCG between 0 and 1 indicates partial success
    """
    idcg = idcg_at_k(qrel, k)
    if idcg == 0:
        return 0.0  # No relevant documents exist — avoid division by zero
    return dcg_at_k(retrieved, qrel, k) / idcg


# ============================================================
# 🏃 STEP 4: Run Evaluation Across All Queries
# ============================================================

def evaluate_all(runs, qrels, k=4):
    """
    Evaluate all queries and compute per-query metrics + aggregated scores.
    
    Returns:
    - per_query: dict of {query: {metric_name: score}} for detailed inspection
    - map_k: Mean Average Precision across all queries
    - mean_ndcg: Mean nDCG across all queries
    """
    per_query = {}
    for q in runs:
        # 📊 Compute all four metrics for this query
        p = precision_at_k(runs[q], qrels[q], k)
        r = recall_at_k(runs[q], qrels[q], k)
        ap = average_precision_at_k(runs[q], qrels[q], k)
        ndcg = ndcg_at_k(runs[q], qrels[q], k)
        per_query[q] = {
            'P@{}'.format(k): p,
            'R@{}'.format(k): r,
            'AP@{}'.format(k): ap,
            'NDCG@{}'.format(k): ndcg
        }

    # 📈 Compute aggregated metrics across all queries
    # MAP = Mean Average Precision (average of AP@k across all queries)
    map_k = statistics.mean([v['AP@{}'.format(k)] for v in per_query.values()])
    # Mean nDCG (average of nDCG@k across all queries)
    mean_ndcg = statistics.mean([v['NDCG@{}'.format(k)] for v in per_query.values()])

    return per_query, map_k, mean_ndcg


# ============================================================
# 🚀 STEP 5: Execute and Display Results
# ============================================================

# Choose k (top-k cutoff). Try changing this to 3, 5, or 10 to see how scores change!
k = 4
per_query, map_k, mean_ndcg = evaluate_all(runs, qrels, k)

# 📊 Pretty-print per-query metrics
print("=" * 60)
print(f"📊 EVALUATION RESULTS (k={k})")
print("=" * 60)

for q, metrics in per_query.items():
    print(f"\n🔍 Query: \"{q}\"")
    for name, val in metrics.items():
        print(f"   {name}: {val:.6f}")

# 📈 Print aggregated metrics
print(f"\n{'=' * 60}")
print(f"📈 AGGREGATED METRICS")
print(f"{'=' * 60}")
print(f"  MAP@{k} (Mean Average Precision): {map_k:.6f}")
print(f"  Mean NDCG@{k}: {mean_ndcg:.6f}")
print(f"\n✅ Evaluation complete!")

📊 EVALUATION RESULTS (k=4)

🔍 Query: "Sweet pastry"
   P@4: 0.500000
   R@4: 0.666667
   AP@4: 0.388889
   NDCG@4: 0.530721

🔍 Query: "Suitable for lunch"
   P@4: 0.500000
   R@4: 0.500000
   AP@4: 0.208333
   NDCG@4: 0.363318

🔍 Query: "Goes well with jam"
   P@4: 0.750000
   R@4: 0.500000
   AP@4: 0.319444
   NDCG@4: 0.413788

📈 AGGREGATED METRICS
  MAP@4 (Mean Average Precision): 0.305556
  Mean NDCG@4: 0.435942

✅ Evaluation complete!


## ✅ Summary & Key Takeaways

### What We Covered 🎉

This notebook provided a **comprehensive journey** through evaluation metrics:

### 📚 Part 1 — Theory:
1. **Traditional Metrics**: Precision, Recall, F1-Score — great for simple retrieval, but insufficient for LLM/RAG systems
2. **RAG-Specific Metrics**: Contextual Precision/Recall, Answer Relevancy, Faithfulness — evaluate both retrieval AND generation
3. **Evaluation Frameworks**: RAGAS (semantic RAG evaluation), G-Eval (LLM-as-judge), DAG (multi-step evaluation)
4. **Agentic & Conversational Metrics**: Task completion, tool correctness, knowledge retention, role adherence

### 💻 Part 2 — Practice:
5. **Implemented from scratch**: Precision@k, Recall@k, AP@k, nDCG@k
6. **Ran evaluations**: Computed per-query and aggregated metrics on bakery-themed example data

### 🧠 Key Concepts to Remember:

| Metric | Best For | Limitation |
|---|---|---|
| **Precision@k** | Quick check of retrieval quality | Doesn't reward good ranking order |
| **Recall@k** | Ensuring completeness | Can be misleading with large k |
| **AP@k / MAP** | Evaluating ranked retrieval | Only handles binary relevance |
| **nDCG@k** | Graded relevance scenarios | More complex to interpret |
| **RAGAS** | Full RAG pipeline evaluation | Requires LLM API calls (cost) |
| **G-Eval** | Open-ended generation quality | Subjective, depends on judge LLM |
| **DAG Evaluation** | Complex multi-criteria assessment | More setup required |

### 🚀 Next Steps:
- **Experiment with different k values** — how do scores change as k increases?
- **Try RAGAS** on a real RAG pipeline (see the `Agent_Evaluation.ipynb` notebook)
- **Build a DAG evaluation** with DeepEval (see the `DAG_EVAL.ipynb` notebook)
- **Compare embedding models** — swap text-embedding-3-small for other models and compare nDCG
- **Create your own qrels** — define ground truth for your domain-specific retrieval tasks

---

*💡 **Pro Tip**: In production, use a combination of metrics — no single metric tells the whole story. MAP@k for retrieval quality, RAGAS faithfulness for hallucination detection, and conversational metrics for multi-turn quality!*